# From-Scratch TTS on Kaggle  
**Transformer Encoder + Tacotron-Style Acoustic Model + HiFi-GAN-Style Vocoder**

This notebook is designed to be:
- **from scratch** (random initialization, no pretrained checkpoints)
- **Kaggle friendly**
- **single-speaker** first (recommended: **LJSpeech**)
- **split into copy-paste cells**

## Why this design?
A full FastSpeech 2 pipeline usually needs reliable phoneme durations from an external aligner or a teacher model.  
For a graduation project on Kaggle, a **Transformer-encoder Tacotron-style acoustic model** is more practical end-to-end, and a **HiFi-GAN-style vocoder** improves final waveform quality substantially.

## Recommended workflow
1. Train the **acoustic model** first until mel predictions are stable.
2. Then train the **vocoder** on ground-truth mels and waveforms.
3. Finally synthesize speech using:
   - text -> predicted mel (acoustic model)
   - mel -> waveform (vocoder)

## Dataset assumption
This notebook assumes LJSpeech is available under one of these Kaggle paths:
- `/kaggle/input/ljspeech/LJSpeech-1.1`
- `/kaggle/input/the-lj-speech-dataset/LJSpeech-1.1`

If your path differs, change `DATA_ROOT` in the config cell.

In [ ]:
# Cell 1: imports
import os
import re
import gc
import math
import json
import random
import pathlib
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchaudio
import matplotlib.pyplot as plt
from IPython.display import Audio, display

In [ ]:
# Cell 2: configuration
@dataclass
class Config:
    # Paths
    data_root_candidates: tuple = (
        "/kaggle/input/ljspeech/LJSpeech-1.1",
        "/kaggle/input/the-lj-speech-dataset/LJSpeech-1.1",
        "/kaggle/input/ljspeech-11/LJSpeech-1.1",
    )
    work_dir: str = "/kaggle/working/tts_from_scratch_transformer_hifigan"
    metadata_name: str = "metadata.csv"

    # Reproducibility
    seed: int = 42

    # Audio
    sample_rate: int = 22050
    n_fft: int = 1024
    win_length: int = 1024
    hop_length: int = 256
    n_mels: int = 80
    f_min: int = 0
    f_max: int = 8000
    preemphasis: float = 0.0
    max_wav_value: float = 32768.0

    # Text
    cleaners_lower: bool = True
    add_bos_eos: bool = True
    max_text_len: int = 220

    # Dataset / batching
    train_size: int | None = None   # set to an int for quick experiments, else None for full usable set
    val_size: int = 300
    batch_size_acoustic: int = 16
    batch_size_vocoder: int = 16
    num_workers: int = 2
    max_wav_len_sec: float = 10.0
    max_mel_len: int = 900

    # Acoustic model
    vocab_embed_dim: int = 256
    encoder_hidden: int = 256
    encoder_layers: int = 4
    encoder_heads: int = 4
    encoder_ffn: int = 1024
    encoder_dropout: float = 0.1

    prenet_dim: int = 256
    attention_rnn_dim: int = 512
    decoder_rnn_dim: int = 512
    attention_dim: int = 128
    attention_location_filters: int = 32
    attention_location_kernel: int = 31
    postnet_channels: int = 512
    max_decoder_steps: int = 1000
    gate_threshold: float = 0.5

    # Training acoustic
    acoustic_epochs: int = 30
    acoustic_lr: float = 2e-4
    acoustic_weight_decay: float = 1e-6
    grad_clip: float = 1.0
    teacher_forcing_ratio: float = 1.0
    teacher_forcing_decay: float = 0.98
    teacher_forcing_min: float = 0.4

    # Vocoder
    upsample_rates: tuple = (8, 8, 2, 2)
    upsample_kernel_sizes: tuple = (16, 16, 4, 4)
    upsample_initial_channel: int = 256
    resblock_kernel_sizes: tuple = (3, 7, 11)
    resblock_dilation_sizes: tuple = ((1, 3, 5), (1, 3, 5), (1, 3, 5))

    # Training vocoder
    vocoder_epochs: int = 60
    vocoder_lr: float = 2e-4
    segment_size: int = 8192
    lambda_mel: float = 45.0
    lambda_fm: float = 2.0

CFG = Config()

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CFG.seed)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def find_data_root(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(
        "Could not find LJSpeech root. Update CFG.data_root_candidates or set DATA_ROOT manually."
    )

DATA_ROOT = find_data_root(CFG.data_root_candidates)
os.makedirs(CFG.work_dir, exist_ok=True)

print("DEVICE:", DEVICE)
print("DATA_ROOT:", DATA_ROOT)
print("WORK_DIR:", CFG.work_dir)

In [ ]:
# Cell 3: text processing
_pad = "_"
_punc = "!'(),-.:;? "
_letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
_digits = "0123456789"

SYMBOLS = [_pad] + list(_punc + _letters + _digits)
PAD_ID = 0

symbol_to_id = {s: i for i, s in enumerate(SYMBOLS)}
id_to_symbol = {i: s for s, i in symbol_to_id.items()}

def basic_cleaner(text: str) -> str:
    text = text.strip()
    text = text.replace("“", '"').replace("”", '"').replace("’", "'")
    text = re.sub(r"\s+", " ", text)
    if CFG.cleaners_lower:
        text = text.lower()
    return text

def text_to_ids(text: str):
    text = basic_cleaner(text)
    ids = []
    if CFG.add_bos_eos and "~" not in symbol_to_id:
        pass
    for ch in text:
        if ch in symbol_to_id:
            ids.append(symbol_to_id[ch])
    ids = ids[:CFG.max_text_len]
    return ids

def ids_to_text(ids):
    return "".join(id_to_symbol.get(i, "") for i in ids if i != PAD_ID)

print("Vocab size:", len(SYMBOLS))
print(text_to_ids("Hello, this is a test 123."))

In [ ]:
# Cell 4: metadata and splits
metadata_path = os.path.join(DATA_ROOT, CFG.metadata_name)
meta = pd.read_csv(metadata_path, sep="|", header=None, names=["id", "text", "normalized_text"])
meta["text"] = meta["normalized_text"].fillna(meta["text"])
meta["wav_path"] = meta["id"].apply(lambda x: os.path.join(DATA_ROOT, "wavs", f"{x}.wav"))
meta = meta[meta["wav_path"].map(os.path.exists)].reset_index(drop=True)

# Filter empty text
meta["text_clean"] = meta["text"].astype(str).apply(basic_cleaner)
meta = meta[meta["text_clean"].str.len() > 0].reset_index(drop=True)

# Optional subset for quick experiments
if CFG.train_size is not None:
    usable = min(len(meta), CFG.train_size + CFG.val_size)
    meta = meta.iloc[:usable].copy().reset_index(drop=True)

val_size = min(CFG.val_size, max(1, len(meta)//20))
train_df = meta.iloc[:-val_size].copy().reset_index(drop=True)
val_df = meta.iloc[-val_size:].copy().reset_index(drop=True)

print("Total usable:", len(meta))
print("Train:", len(train_df))
print("Val:", len(val_df))
train_df.head()

In [ ]:
# Cell 5: audio processor
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=CFG.sample_rate,
    n_fft=CFG.n_fft,
    win_length=CFG.win_length,
    hop_length=CFG.hop_length,
    f_min=CFG.f_min,
    f_max=CFG.f_max,
    n_mels=CFG.n_mels,
    power=1.0,
    center=True,
).to(DEVICE)

amp_to_db = torchaudio.transforms.AmplitudeToDB(stype="power").to(DEVICE)

def load_wav(path):
    wav, sr = torchaudio.load(path)
    wav = wav.mean(dim=0, keepdim=True)
    if sr != CFG.sample_rate:
        wav = torchaudio.functional.resample(wav, sr, CFG.sample_rate)
    wav = wav.squeeze(0)
    return wav

def wav_to_mel(wav_1d: torch.Tensor):
    x = wav_1d.unsqueeze(0).to(DEVICE)
    mel = mel_transform(x)
    mel = torch.log(torch.clamp(mel, min=1e-5))
    return mel.squeeze(0).cpu()

@torch.no_grad()
def debug_audio(path):
    wav = load_wav(path)
    mel = wav_to_mel(wav)
    print("wav:", wav.shape, "mel:", mel.shape)
    plt.figure(figsize=(12, 4))
    plt.imshow(mel.numpy(), aspect="auto", origin="lower")
    plt.title("Mel Spectrogram")
    plt.tight_layout()
    plt.show()
    display(Audio(wav.numpy(), rate=CFG.sample_rate))

debug_audio(train_df.iloc[0]["wav_path"])

In [ ]:
# Cell 6: dataset and collate
class LJTTSDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text_ids = text_to_ids(row["text_clean"])
        wav = load_wav(row["wav_path"])

        max_wav_len = int(CFG.max_wav_len_sec * CFG.sample_rate)
        if wav.numel() > max_wav_len:
            wav = wav[:max_wav_len]

        mel = wav_to_mel(wav)
        if mel.size(1) > CFG.max_mel_len:
            mel = mel[:, :CFG.max_mel_len]
            wav = wav[: CFG.max_mel_len * CFG.hop_length]

        return {
            "id": row["id"],
            "text_ids": torch.tensor(text_ids, dtype=torch.long),
            "mel": mel.float(),
            "wav": wav.float(),
        }

def pad_1d(seqs, pad_value=0):
    max_len = max(x.size(0) for x in seqs)
    out = torch.full((len(seqs), max_len), pad_value, dtype=seqs[0].dtype)
    lengths = []
    for i, x in enumerate(seqs):
        out[i, :x.size(0)] = x
        lengths.append(x.size(0))
    return out, torch.tensor(lengths, dtype=torch.long)

def pad_mels(mels):
    max_len = max(m.size(1) for m in mels)
    out = torch.zeros(len(mels), CFG.n_mels, max_len)
    gate = torch.ones(len(mels), max_len)
    lengths = []
    for i, m in enumerate(mels):
        T = m.size(1)
        out[i, :, :T] = m
        gate[i, :T-1] = 0.0
        gate[i, T-1:] = 1.0
        lengths.append(T)
    return out, gate, torch.tensor(lengths, dtype=torch.long)

def pad_wavs(wavs):
    max_len = max(w.size(0) for w in wavs)
    out = torch.zeros(len(wavs), max_len)
    lengths = []
    for i, w in enumerate(wavs):
        out[i, :w.size(0)] = w
        lengths.append(w.size(0))
    return out, torch.tensor(lengths, dtype=torch.long)

def collate_fn(batch):
    texts = [b["text_ids"] for b in batch]
    mels = [b["mel"] for b in batch]
    wavs = [b["wav"] for b in batch]
    ids = [b["id"] for b in batch]

    text_pad, text_lens = pad_1d(texts, PAD_ID)
    mel_pad, gate_pad, mel_lens = pad_mels(mels)
    wav_pad, wav_lens = pad_wavs(wavs)

    # sort by text length desc helps some RNN stability
    order = torch.argsort(text_lens, descending=True)
    return {
        "ids": [ids[i] for i in order.tolist()],
        "text": text_pad[order],
        "text_lens": text_lens[order],
        "mel": mel_pad[order],
        "gate": gate_pad[order],
        "mel_lens": mel_lens[order],
        "wav": wav_pad[order],
        "wav_lens": wav_lens[order],
    }

train_ds = LJTTSDataset(train_df)
val_ds = LJTTSDataset(val_df)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG.batch_size_acoustic,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=True,
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    val_ds,
    batch_size=CFG.batch_size_acoustic,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=True,
    collate_fn=collate_fn,
)

batch = next(iter(train_loader))
for k, v in batch.items():
    if torch.is_tensor(v):
        print(k, tuple(v.shape))
    else:
        print(k, type(v), len(v))

In [ ]:
# Cell 7: positional encoding + utility masks
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        pe = pe.unsqueeze(0)  # [1, T, D]
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

def get_mask_from_lengths(lengths, max_len=None):
    if max_len is None:
        max_len = lengths.max().item()
    ids = torch.arange(0, max_len, device=lengths.device)
    return ids.unsqueeze(0) >= lengths.unsqueeze(1)

In [ ]:
# Cell 8: acoustic model blocks
class Prenet(nn.Module):
    def __init__(self, in_dim, sizes=(256, 256), dropout=0.5):
        super().__init__()
        layers = []
        prev = in_dim
        for s in sizes:
            layers += [nn.Linear(prev, s), nn.ReLU()]
            prev = s
        self.layers = nn.ModuleList(layers)
        self.dropout = dropout

    def forward(self, x):
        for i in range(0, len(self.layers), 2):
            x = self.layers[i](x)
            x = self.layers[i+1](x)
            x = F.dropout(x, p=self.dropout, training=True)
        return x

class Postnet(nn.Module):
    def __init__(self, n_mels=80, channels=512, kernel_size=5, n_convs=5, dropout=0.5):
        super().__init__()
        layers = []
        in_ch = n_mels
        for i in range(n_convs):
            out_ch = channels if i < n_convs - 1 else n_mels
            conv = nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=(kernel_size-1)//2)
            bn = nn.BatchNorm1d(out_ch)
            layers.append(nn.Sequential(conv, bn))
            in_ch = out_ch
        self.layers = nn.ModuleList(layers)
        self.dropout = dropout

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            x = torch.tanh(x) if i < len(self.layers)-1 else x
            x = F.dropout(x, p=self.dropout, training=self.training)
        return x

class LocationLayer(nn.Module):
    def __init__(self, attention_n_filters, attention_kernel_size, attention_dim):
        super().__init__()
        padding = (attention_kernel_size - 1) // 2
        self.location_conv = nn.Conv1d(
            2, attention_n_filters, kernel_size=attention_kernel_size, padding=padding, bias=False
        )
        self.location_dense = nn.Linear(attention_n_filters, attention_dim, bias=False)

    def forward(self, attention_weights_cat):
        processed = self.location_conv(attention_weights_cat)
        processed = processed.transpose(1, 2)
        return self.location_dense(processed)

class Attention(nn.Module):
    def __init__(self, attention_rnn_dim, encoder_dim, attention_dim, attention_location_n_filters, attention_location_kernel_size):
        super().__init__()
        self.query_layer = nn.Linear(attention_rnn_dim, attention_dim, bias=False)
        self.memory_layer = nn.Linear(encoder_dim, attention_dim, bias=False)
        self.v = nn.Linear(attention_dim, 1, bias=False)
        self.location_layer = LocationLayer(attention_location_n_filters, attention_location_kernel_size, attention_dim)
        self.score_mask_value = -float("inf")

    def get_alignment_energies(self, query, processed_memory, attention_weights_cat):
        processed_query = self.query_layer(query).unsqueeze(1)
        processed_attention_weights = self.location_layer(attention_weights_cat)
        energies = self.v(torch.tanh(processed_query + processed_attention_weights + processed_memory)).squeeze(-1)
        return energies

    def forward(self, attention_hidden_state, memory, processed_memory, attention_weights_cat, mask):
        alignment = self.get_alignment_energies(attention_hidden_state, processed_memory, attention_weights_cat)
        if mask is not None:
            alignment.data.masked_fill_(mask, self.score_mask_value)
        attention_weights = F.softmax(alignment, dim=1)
        attention_context = torch.bmm(attention_weights.unsqueeze(1), memory).squeeze(1)
        return attention_context, attention_weights

In [ ]:
# Cell 9: acoustic model
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, n_heads, ffn_dim, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        self.in_proj = nn.Linear(embed_dim, hidden_dim)
        self.pos_enc = PositionalEncoding(hidden_dim, dropout=dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=n_heads,
            dim_feedforward=ffn_dim,
            dropout=dropout,
            batch_first=True,
            activation="relu",
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

    def forward(self, text, text_lens):
        x = self.embedding(text)
        x = self.in_proj(x)
        x = self.pos_enc(x)
        src_key_padding_mask = get_mask_from_lengths(text_lens, text.size(1)).to(text.device)
        out = self.encoder(x, src_key_padding_mask=src_key_padding_mask)
        return out, src_key_padding_mask

class AcousticModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = TransformerEncoder(
            vocab_size=len(SYMBOLS),
            embed_dim=CFG.vocab_embed_dim,
            hidden_dim=CFG.encoder_hidden,
            n_layers=CFG.encoder_layers,
            n_heads=CFG.encoder_heads,
            ffn_dim=CFG.encoder_ffn,
            dropout=CFG.encoder_dropout,
        )
        encoder_dim = CFG.encoder_hidden
        self.prenet = Prenet(CFG.n_mels, sizes=(CFG.prenet_dim, CFG.prenet_dim))
        self.attention_rnn = nn.LSTMCell(CFG.prenet_dim + encoder_dim, CFG.attention_rnn_dim)
        self.attention_layer = Attention(
            CFG.attention_rnn_dim, encoder_dim, CFG.attention_dim,
            CFG.attention_location_filters, CFG.attention_location_kernel
        )
        self.decoder_rnn = nn.LSTMCell(CFG.attention_rnn_dim + encoder_dim, CFG.decoder_rnn_dim)
        self.mel_proj = nn.Linear(CFG.decoder_rnn_dim + encoder_dim, CFG.n_mels)
        self.gate_proj = nn.Linear(CFG.decoder_rnn_dim + encoder_dim, 1)
        self.postnet = Postnet(CFG.n_mels, CFG.postnet_channels)

    def initialize_decoder_states(self, memory):
        B = memory.size(0)
        device = memory.device
        encoder_dim = memory.size(2)

        attn_hidden = torch.zeros(B, CFG.attention_rnn_dim, device=device)
        attn_cell = torch.zeros(B, CFG.attention_rnn_dim, device=device)
        dec_hidden = torch.zeros(B, CFG.decoder_rnn_dim, device=device)
        dec_cell = torch.zeros(B, CFG.decoder_rnn_dim, device=device)
        attn_weights = torch.zeros(B, memory.size(1), device=device)
        attn_weights_cum = torch.zeros(B, memory.size(1), device=device)
        attn_context = torch.zeros(B, encoder_dim, device=device)

        return attn_hidden, attn_cell, dec_hidden, dec_cell, attn_weights, attn_weights_cum, attn_context

    def parse_decoder_inputs(self, mels):
        return mels.transpose(1, 2)

    def decode_step(self, decoder_input, states, memory, processed_memory, mask):
        (attn_hidden, attn_cell, dec_hidden, dec_cell, attn_weights, attn_weights_cum, attn_context) = states

        cell_input = torch.cat([decoder_input, attn_context], dim=-1)
        attn_hidden, attn_cell = self.attention_rnn(cell_input, (attn_hidden, attn_cell))

        attn_weights_cat = torch.cat(
            [attn_weights.unsqueeze(1), attn_weights_cum.unsqueeze(1)], dim=1
        )
        attn_context, attn_weights = self.attention_layer(
            attn_hidden, memory, processed_memory, attn_weights_cat, mask
        )
        attn_weights_cum = attn_weights_cum + attn_weights

        decoder_input2 = torch.cat([attn_hidden, attn_context], dim=-1)
        dec_hidden, dec_cell = self.decoder_rnn(decoder_input2, (dec_hidden, dec_cell))

        decoder_hidden_attention_context = torch.cat([dec_hidden, attn_context], dim=1)
        mel_out = self.mel_proj(decoder_hidden_attention_context)
        gate_out = self.gate_proj(decoder_hidden_attention_context).squeeze(-1)

        new_states = (attn_hidden, attn_cell, dec_hidden, dec_cell, attn_weights, attn_weights_cum, attn_context)
        return mel_out, gate_out, new_states, attn_weights

    def forward(self, text, text_lens, mels, teacher_forcing_ratio=1.0):
        memory, mask = self.encoder(text, text_lens)
        processed_memory = self.attention_layer.memory_layer(memory)

        decoder_inputs = self.parse_decoder_inputs(mels)
        go_frame = torch.zeros(mels.size(0), CFG.n_mels, device=mels.device)
        states = self.initialize_decoder_states(memory)

        mel_outputs, gate_outputs, alignments = [], [], []
        decoder_input = go_frame

        T_out = decoder_inputs.size(1)
        for t in range(T_out):
            prenet_out = self.prenet(decoder_input)
            mel_out, gate_out, states, align = self.decode_step(prenet_out, states, memory, processed_memory, mask)
            mel_outputs.append(mel_out.unsqueeze(1))
            gate_outputs.append(gate_out.unsqueeze(1))
            alignments.append(align.unsqueeze(1))

            use_teacher = random.random() < teacher_forcing_ratio
            decoder_input = decoder_inputs[:, t, :] if use_teacher else mel_out

        mel_outputs = torch.cat(mel_outputs, dim=1).transpose(1, 2)
        gate_outputs = torch.cat(gate_outputs, dim=1).squeeze(-1)
        alignments = torch.cat(alignments, dim=1)

        mel_post = mel_outputs + self.postnet(mel_outputs)
        return mel_outputs, mel_post, gate_outputs, alignments

    @torch.no_grad()
    def infer(self, text, text_lens, max_steps=None):
        self.eval()
        if max_steps is None:
            max_steps = CFG.max_decoder_steps

        memory, mask = self.encoder(text, text_lens)
        processed_memory = self.attention_layer.memory_layer(memory)
        states = self.initialize_decoder_states(memory)

        decoder_input = torch.zeros(text.size(0), CFG.n_mels, device=text.device)
        mel_outputs, gate_outputs, alignments = [], [], []

        for _ in range(max_steps):
            prenet_out = self.prenet(decoder_input)
            mel_out, gate_out, states, align = self.decode_step(prenet_out, states, memory, processed_memory, mask)

            mel_outputs.append(mel_out.unsqueeze(1))
            gate_outputs.append(gate_out.unsqueeze(1))
            alignments.append(align.unsqueeze(1))

            decoder_input = mel_out
            if torch.sigmoid(gate_out).item() > CFG.gate_threshold and len(mel_outputs) > 10:
                break

        mel_outputs = torch.cat(mel_outputs, dim=1).transpose(1, 2)
        mel_post = mel_outputs + self.postnet(mel_outputs)
        gate_outputs = torch.cat(gate_outputs, dim=1).squeeze(-1)
        alignments = torch.cat(alignments, dim=1)
        return mel_outputs, mel_post, gate_outputs, alignments

acoustic_model = AcousticModel().to(DEVICE)
num_params = sum(p.numel() for p in acoustic_model.parameters())
print("Acoustic params:", f"{num_params:,}")

In [ ]:
# Cell 10: acoustic losses and helpers
def acoustic_loss_fn(mel_out, mel_post, gate_out, mel_target, gate_target):
    mel_loss = F.l1_loss(mel_out, mel_target) + F.l1_loss(mel_post, mel_target)
    gate_loss = F.binary_cross_entropy_with_logits(gate_out, gate_target)
    return mel_loss, gate_loss, mel_loss + gate_loss

def move_batch(batch):
    out = {}
    for k, v in batch.items():
        out[k] = v.to(DEVICE) if torch.is_tensor(v) else v
    return out

def plot_alignment(alignment, title="Alignment"):
    plt.figure(figsize=(8, 6))
    plt.imshow(alignment, aspect="auto", origin="lower")
    plt.title(title)
    plt.xlabel("Encoder steps")
    plt.ylabel("Decoder steps")
    plt.tight_layout()
    plt.show()

def plot_mel(mel, title="Mel"):
    plt.figure(figsize=(12, 4))
    plt.imshow(mel, aspect="auto", origin="lower")
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 11: acoustic training
acoustic_optimizer = torch.optim.AdamW(
    acoustic_model.parameters(),
    lr=CFG.acoustic_lr,
    weight_decay=CFG.acoustic_weight_decay,
)
acoustic_scheduler = torch.optim.lr_scheduler.ExponentialLR(acoustic_optimizer, gamma=0.98)

best_val_acoustic = float("inf")
teacher_forcing_ratio = CFG.teacher_forcing_ratio
acoustic_ckpt_path = os.path.join(CFG.work_dir, "best_acoustic.pt")
history_acoustic = []

for epoch in range(1, CFG.acoustic_epochs + 1):
    acoustic_model.train()
    train_losses = []

    pbar = tqdm(train_loader, desc=f"Acoustic Epoch {epoch}/{CFG.acoustic_epochs}")
    for batch in pbar:
        batch = move_batch(batch)

        acoustic_optimizer.zero_grad()
        mel_out, mel_post, gate_out, align = acoustic_model(
            batch["text"], batch["text_lens"], batch["mel"],
            teacher_forcing_ratio=teacher_forcing_ratio
        )
        mel_loss, gate_loss, total_loss = acoustic_loss_fn(
            mel_out, mel_post, gate_out, batch["mel"], batch["gate"]
        )
        total_loss.backward()
        nn.utils.clip_grad_norm_(acoustic_model.parameters(), CFG.grad_clip)
        acoustic_optimizer.step()

        train_losses.append(total_loss.item())
        pbar.set_postfix(
            total=f"{np.mean(train_losses):.4f}",
            mel=f"{mel_loss.item():.4f}",
            gate=f"{gate_loss.item():.4f}",
            tf=f"{teacher_forcing_ratio:.2f}",
        )

    acoustic_model.eval()
    val_losses = []
    with torch.no_grad():
        for batch in val_loader:
            batch = move_batch(batch)
            mel_out, mel_post, gate_out, align = acoustic_model(
                batch["text"], batch["text_lens"], batch["mel"], teacher_forcing_ratio=1.0
            )
            mel_loss, gate_loss, total_loss = acoustic_loss_fn(
                mel_out, mel_post, gate_out, batch["mel"], batch["gate"]
            )
            val_losses.append(total_loss.item())

    train_loss = float(np.mean(train_losses))
    val_loss = float(np.mean(val_losses))
    history_acoustic.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})

    if val_loss < best_val_acoustic:
        best_val_acoustic = val_loss
        torch.save(
            {
                "model": acoustic_model.state_dict(),
                "cfg": asdict(CFG),
                "history": history_acoustic,
                "best_val": best_val_acoustic,
            },
            acoustic_ckpt_path
        )
        print(f"Saved best acoustic checkpoint to {acoustic_ckpt_path}")

    print(f"Epoch {epoch:02d} | train {train_loss:.4f} | val {val_loss:.4f}")

    # preview a validation sample
    preview = next(iter(val_loader))
    preview = move_batch(preview)
    mel_out, mel_post, gate_out, align = acoustic_model(
        preview["text"][:1], preview["text_lens"][:1], preview["mel"][:1], teacher_forcing_ratio=1.0
    )
    plot_alignment(align[0].detach().cpu().numpy(), title=f"Val Alignment Epoch {epoch}")
    plot_mel(preview["mel"][0].detach().cpu().numpy(), title="Ground Truth Mel")
    plot_mel(mel_post[0].detach().cpu().numpy(), title="Predicted Mel (Postnet)")

    acoustic_scheduler.step()
    teacher_forcing_ratio = max(
        CFG.teacher_forcing_min,
        teacher_forcing_ratio * CFG.teacher_forcing_decay
    )
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# Cell 12: load best acoustic model
ckpt = torch.load(acoustic_ckpt_path, map_location=DEVICE)
acoustic_model.load_state_dict(ckpt["model"])
acoustic_model.eval()
print("Best acoustic val loss:", ckpt["best_val"])

In [ ]:
# Cell 13: acoustic inference sanity check
@torch.no_grad()
def text_to_mel_infer(model, text_str):
    ids = text_to_ids(text_str)
    x = torch.tensor(ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
    lens = torch.tensor([len(ids)], dtype=torch.long, device=DEVICE)
    mel_out, mel_post, gate_out, align = model.infer(x, lens)
    return mel_post[0].cpu(), align[0].cpu()

test_text = "Hello, this is my graduation project."
pred_mel, pred_align = text_to_mel_infer(acoustic_model, test_text)
print("Pred mel shape:", pred_mel.shape)
plot_alignment(pred_align.numpy(), title="Inference Alignment")
plot_mel(pred_mel.numpy(), title="Predicted Mel")

In [ ]:
# Cell 14: vocoder dataset helper
class VocoderDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        wav = load_wav(row["wav_path"])
        mel = wav_to_mel(wav)

        segment = CFG.segment_size
        mel_segment_frames = segment // CFG.hop_length

        if wav.size(0) >= segment and mel.size(1) >= mel_segment_frames:
            max_mel_start = mel.size(1) - mel_segment_frames
            mel_start = random.randint(0, max_mel_start)
            wav_start = mel_start * CFG.hop_length
            wav = wav[wav_start:wav_start + segment]
            mel = mel[:, mel_start:mel_start + mel_segment_frames]
        else:
            if wav.size(0) < segment:
                wav = F.pad(wav, (0, segment - wav.size(0)))
            if mel.size(1) < mel_segment_frames:
                mel = F.pad(mel, (0, mel_segment_frames - mel.size(1)))

        return mel.float(), wav.float().unsqueeze(0)

def vocoder_collate(batch):
    mels = torch.stack([b[0] for b in batch], dim=0)
    wavs = torch.stack([b[1] for b in batch], dim=0)
    return mels, wavs

voc_train_loader = DataLoader(
    VocoderDataset(train_df),
    batch_size=CFG.batch_size_vocoder,
    shuffle=True,
    num_workers=CFG.num_workers,
    pin_memory=True,
    collate_fn=vocoder_collate,
)
voc_val_loader = DataLoader(
    VocoderDataset(val_df),
    batch_size=CFG.batch_size_vocoder,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=True,
    collate_fn=vocoder_collate,
)

mels, wavs = next(iter(voc_train_loader))
print(mels.shape, wavs.shape)

In [ ]:
# Cell 15: HiFi-GAN-style vocoder blocks
LRELU_SLOPE = 0.1

class ResBlock1(nn.Module):
    def __init__(self, channels, kernel_size=3, dilations=(1, 3, 5)):
        super().__init__()
        self.convs1 = nn.ModuleList([
            nn.utils.weight_norm(nn.Conv1d(channels, channels, kernel_size, 1, dilation=d, padding=((kernel_size*d)-d)//2))
            for d in dilations
        ])
        self.convs2 = nn.ModuleList([
            nn.utils.weight_norm(nn.Conv1d(channels, channels, kernel_size, 1, dilation=1, padding=(kernel_size-1)//2))
            for _ in dilations
        ])

    def forward(self, x):
        for c1, c2 in zip(self.convs1, self.convs2):
            xt = F.leaky_relu(x, LRELU_SLOPE)
            xt = c1(xt)
            xt = F.leaky_relu(xt, LRELU_SLOPE)
            xt = c2(xt)
            x = xt + x
        return x

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_pre = nn.utils.weight_norm(nn.Conv1d(CFG.n_mels, CFG.upsample_initial_channel, 7, 1, padding=3))

        self.ups = nn.ModuleList()
        self.resblocks = nn.ModuleList()

        cur_ch = CFG.upsample_initial_channel
        for u, k in zip(CFG.upsample_rates, CFG.upsample_kernel_sizes):
            self.ups.append(
                nn.utils.weight_norm(
                    nn.ConvTranspose1d(cur_ch, cur_ch // 2, k, u, padding=(k - u) // 2)
                )
            )
            cur_ch = cur_ch // 2
            for rk, rd in zip(CFG.resblock_kernel_sizes, CFG.resblock_dilation_sizes):
                self.resblocks.append(ResBlock1(cur_ch, kernel_size=rk, dilations=rd))

        self.num_kernels = len(CFG.resblock_kernel_sizes)
        self.conv_post = nn.utils.weight_norm(nn.Conv1d(cur_ch, 1, 7, 1, padding=3))

    def forward(self, x):
        x = self.conv_pre(x)
        rb_idx = 0
        for up in self.ups:
            x = F.leaky_relu(x, LRELU_SLOPE)
            x = up(x)
            xs = 0
            for _ in range(self.num_kernels):
                xs = xs + self.resblocks[rb_idx](x)
                rb_idx += 1
            x = xs / self.num_kernels
        x = F.leaky_relu(x, LRELU_SLOPE)
        x = self.conv_post(x)
        x = torch.tanh(x)
        return x

class DiscriminatorP(nn.Module):
    def __init__(self, period):
        super().__init__()
        self.period = period
        self.convs = nn.ModuleList([
            nn.utils.weight_norm(nn.Conv2d(1, 32, (5, 1), (3, 1), padding=(2, 0))),
            nn.utils.weight_norm(nn.Conv2d(32, 128, (5, 1), (3, 1), padding=(2, 0))),
            nn.utils.weight_norm(nn.Conv2d(128, 512, (5, 1), (3, 1), padding=(2, 0))),
            nn.utils.weight_norm(nn.Conv2d(512, 1024, (5, 1), 1, padding=(2, 0))),
            nn.utils.weight_norm(nn.Conv2d(1024, 1024, (5, 1), 1, padding=(2, 0))),
        ])
        self.conv_post = nn.utils.weight_norm(nn.Conv2d(1024, 1, (3, 1), 1, padding=(1, 0)))

    def forward(self, x):
        fmap = []
        b, c, t = x.shape
        if t % self.period != 0:
            n_pad = self.period - (t % self.period)
            x = F.pad(x, (0, n_pad), "reflect")
            t = t + n_pad
        x = x.view(b, c, t // self.period, self.period)

        for l in self.convs:
            x = l(x)
            x = F.leaky_relu(x, LRELU_SLOPE)
            fmap.append(x)
        x = self.conv_post(x)
        fmap.append(x)
        x = torch.flatten(x, 1, -1)
        return x, fmap

class MultiPeriodDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.discriminators = nn.ModuleList([DiscriminatorP(p) for p in [2, 3, 5, 7, 11]])

    def forward(self, y, y_hat):
        y_d_rs, y_d_gs, fmap_rs, fmap_gs = [], [], [], []
        for d in self.discriminators:
            y_r, fmap_r = d(y)
            y_g, fmap_g = d(y_hat)
            y_d_rs.append(y_r)
            y_d_gs.append(y_g)
            fmap_rs.append(fmap_r)
            fmap_gs.append(fmap_g)
        return y_d_rs, y_d_gs, fmap_rs, fmap_gs

class DiscriminatorS(nn.Module):
    def __init__(self):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.utils.weight_norm(nn.Conv1d(1, 128, 15, 1, padding=7)),
            nn.utils.weight_norm(nn.Conv1d(128, 128, 41, 2, groups=4, padding=20)),
            nn.utils.weight_norm(nn.Conv1d(128, 256, 41, 2, groups=16, padding=20)),
            nn.utils.weight_norm(nn.Conv1d(256, 512, 41, 4, groups=16, padding=20)),
            nn.utils.weight_norm(nn.Conv1d(512, 1024, 41, 4, groups=16, padding=20)),
            nn.utils.weight_norm(nn.Conv1d(1024, 1024, 41, 1, groups=16, padding=20)),
            nn.utils.weight_norm(nn.Conv1d(1024, 1024, 5, 1, padding=2)),
        ])
        self.conv_post = nn.utils.weight_norm(nn.Conv1d(1024, 1, 3, 1, padding=1))

    def forward(self, x):
        fmap = []
        for l in self.convs:
            x = l(x)
            x = F.leaky_relu(x, LRELU_SLOPE)
            fmap.append(x)
        x = self.conv_post(x)
        fmap.append(x)
        x = torch.flatten(x, 1, -1)
        return x, fmap

class MultiScaleDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.discriminators = nn.ModuleList([DiscriminatorS(), DiscriminatorS(), DiscriminatorS()])
        self.pooling = nn.ModuleList([
            nn.Identity(),
            nn.AvgPool1d(4, 2, padding=2),
            nn.AvgPool1d(4, 2, padding=2),
        ])

    def forward(self, y, y_hat):
        y_d_rs, y_d_gs, fmap_rs, fmap_gs = [], [], [], []
        yr, yg = y, y_hat
        for i, d in enumerate(self.discriminators):
            if i > 0:
                yr = self.pooling[i](yr)
                yg = self.pooling[i](yg)
            y_r, fmap_r = d(yr)
            y_g, fmap_g = d(yg)
            y_d_rs.append(y_r)
            y_d_gs.append(y_g)
            fmap_rs.append(fmap_r)
            fmap_gs.append(fmap_g)
        return y_d_rs, y_d_gs, fmap_rs, fmap_gs

generator = Generator().to(DEVICE)
mpd = MultiPeriodDiscriminator().to(DEVICE)
msd = MultiScaleDiscriminator().to(DEVICE)

print("Generator params:", f"{sum(p.numel() for p in generator.parameters()):,}")

In [ ]:
# Cell 16: vocoder losses
def feature_loss(fmap_r, fmap_g):
    loss = 0
    for dr, dg in zip(fmap_r, fmap_g):
        for rl, gl in zip(dr, dg):
            loss += torch.mean(torch.abs(rl - gl))
    return loss * 2

def discriminator_loss(disc_real_outputs, disc_generated_outputs):
    loss = 0
    real_losses = []
    gen_losses = []
    for dr, dg in zip(disc_real_outputs, disc_generated_outputs):
        r_loss = torch.mean((1 - dr) ** 2)
        g_loss = torch.mean(dg ** 2)
        loss += (r_loss + g_loss)
        real_losses.append(r_loss.item())
        gen_losses.append(g_loss.item())
    return loss, real_losses, gen_losses

def generator_loss(disc_outputs):
    loss = 0
    gen_losses = []
    for dg in disc_outputs:
        l = torch.mean((1 - dg) ** 2)
        gen_losses.append(l)
        loss += l
    return loss, gen_losses

def mel_reconstruction_loss(y_hat, y):
    # y_hat, y are [B,1,T]
    y_hat = y_hat.squeeze(1)
    y = y.squeeze(1)
    m1 = []
    m2 = []
    for i in range(y.size(0)):
        m1.append(wav_to_mel(y_hat[i].detach().cpu()).to(DEVICE))
        m2.append(wav_to_mel(y[i].detach().cpu()).to(DEVICE))
    m1 = torch.stack(m1, 0)
    m2 = torch.stack(m2, 0)
    return F.l1_loss(m1, m2)

In [ ]:
# Cell 17: vocoder training
optim_g = torch.optim.AdamW(generator.parameters(), lr=CFG.vocoder_lr, betas=(0.8, 0.99))
optim_d = torch.optim.AdamW(
    list(mpd.parameters()) + list(msd.parameters()),
    lr=CFG.vocoder_lr, betas=(0.8, 0.99)
)

sched_g = torch.optim.lr_scheduler.ExponentialLR(optim_g, gamma=0.999)
sched_d = torch.optim.lr_scheduler.ExponentialLR(optim_d, gamma=0.999)

best_vocoder = float("inf")
vocoder_ckpt_path = os.path.join(CFG.work_dir, "best_vocoder.pt")
history_vocoder = []

for epoch in range(1, CFG.vocoder_epochs + 1):
    generator.train(); mpd.train(); msd.train()
    running_g, running_d = [], []

    pbar = tqdm(voc_train_loader, desc=f"Vocoder Epoch {epoch}/{CFG.vocoder_epochs}")
    for mels, wavs in pbar:
        mels = mels.to(DEVICE)
        wavs = wavs.to(DEVICE)

        # Generator forward
        y_hat = generator(mels)

        # ---- Discriminator step ----
        optim_d.zero_grad()
        y_df_r, y_df_g, _, _ = mpd(wavs, y_hat.detach())
        loss_disc_f, _, _ = discriminator_loss(y_df_r, y_df_g)
        y_ds_r, y_ds_g, _, _ = msd(wavs, y_hat.detach())
        loss_disc_s, _, _ = discriminator_loss(y_ds_r, y_ds_g)
        loss_disc_all = loss_disc_f + loss_disc_s
        loss_disc_all.backward()
        optim_d.step()

        # ---- Generator step ----
        optim_g.zero_grad()
        y_df_r, y_df_g, fmap_f_r, fmap_f_g = mpd(wavs, y_hat)
        y_ds_r, y_ds_g, fmap_s_r, fmap_s_g = msd(wavs, y_hat)

        loss_fm_f = feature_loss(fmap_f_r, fmap_f_g)
        loss_fm_s = feature_loss(fmap_s_r, fmap_s_g)
        loss_gen_f, _ = generator_loss(y_df_g)
        loss_gen_s, _ = generator_loss(y_ds_g)
        loss_mel = mel_reconstruction_loss(y_hat, wavs)

        loss_gen_all = loss_gen_f + loss_gen_s + CFG.lambda_fm * (loss_fm_f + loss_fm_s) + CFG.lambda_mel * loss_mel
        loss_gen_all.backward()
        optim_g.step()

        running_g.append(loss_gen_all.item())
        running_d.append(loss_disc_all.item())
        pbar.set_postfix(g=f"{np.mean(running_g):.4f}", d=f"{np.mean(running_d):.4f}", mel=f"{loss_mel.item():.4f}")

    # small validation using mel loss only
    generator.eval()
    val_mel_losses = []
    with torch.no_grad():
        for mels, wavs in voc_val_loader:
            mels = mels.to(DEVICE)
            wavs = wavs.to(DEVICE)
            y_hat = generator(mels)
            val_mel_losses.append(mel_reconstruction_loss(y_hat, wavs).item())
    val_metric = float(np.mean(val_mel_losses))
    history_vocoder.append({"epoch": epoch, "train_g": float(np.mean(running_g)), "train_d": float(np.mean(running_d)), "val_mel": val_metric})

    if val_metric < best_vocoder:
        best_vocoder = val_metric
        torch.save({
            "generator": generator.state_dict(),
            "mpd": mpd.state_dict(),
            "msd": msd.state_dict(),
            "cfg": asdict(CFG),
            "history": history_vocoder,
            "best_val_mel": best_vocoder,
        }, vocoder_ckpt_path)
        print(f"Saved best vocoder checkpoint to {vocoder_ckpt_path}")

    print(f"Epoch {epoch:02d} | G {np.mean(running_g):.4f} | D {np.mean(running_d):.4f} | ValMel {val_metric:.4f}")
    sched_g.step()
    sched_d.step()
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# Cell 18: load best vocoder
voc_ckpt = torch.load(vocoder_ckpt_path, map_location=DEVICE)
generator.load_state_dict(voc_ckpt["generator"])
generator.eval()
print("Best vocoder val mel:", voc_ckpt["best_val_mel"])

In [ ]:
# Cell 19: waveform generation utilities
@torch.no_grad()
def mel_to_audio_infer(generator, mel):
    if mel.dim() == 2:
        mel = mel.unsqueeze(0)
    mel = mel.to(DEVICE)
    wav = generator(mel).squeeze(0).squeeze(0).detach().cpu()
    return wav

@torch.no_grad()
def synthesize(text_str):
    mel, align = text_to_mel_infer(acoustic_model, text_str)
    wav = mel_to_audio_infer(generator, mel)
    return mel, align, wav

sample_text = "Hello, this is my graduation project from scratch."
mel, align, wav = synthesize(sample_text)
print("Mel:", mel.shape, "Wav:", wav.shape)
plot_alignment(align.numpy(), title="Final Inference Alignment")
plot_mel(mel.numpy(), title="Final Predicted Mel")
display(Audio(wav.numpy(), rate=CFG.sample_rate))

In [ ]:
# Cell 20: save demo outputs
demo_dir = os.path.join(CFG.work_dir, "demo_outputs")
os.makedirs(demo_dir, exist_ok=True)

demo_texts = [
    "Hello, this is my graduation project from scratch.",
    "Speech synthesis is challenging but very interesting.",
    "I trained this model on L J Speech using a Kaggle G P U.",
]

results = []
for i, txt in enumerate(demo_texts, 1):
    mel, align, wav = synthesize(txt)
    wav_path = os.path.join(demo_dir, f"demo_{i:02d}.wav")
    img_path = os.path.join(demo_dir, f"demo_{i:02d}_mel.png")
    torchaudio.save(wav_path, wav.unsqueeze(0), CFG.sample_rate)

    plt.figure(figsize=(12, 4))
    plt.imshow(mel.numpy(), aspect="auto", origin="lower")
    plt.title(txt)
    plt.tight_layout()
    plt.savefig(img_path, bbox_inches="tight")
    plt.close()

    results.append({"text": txt, "wav_path": wav_path, "mel_plot": img_path})

pd.DataFrame(results)

## Notes you should put in your report

### Is this still “from scratch”?
Yes, because:
- all model weights are initialized randomly
- training is done on your dataset
- no pretrained TTS checkpoints are loaded
- only standard PyTorch / torchaudio building blocks are used

### Why not merge datasets immediately?
For a first meaningful result, **single-speaker LJSpeech** is the safest option.  
Mixing datasets like LJSpeech and VCTK turns the problem into a **multi-speaker** task and usually needs:
- speaker IDs / speaker embeddings
- stronger normalization
- more training time
- more careful evaluation

### What to try first if results are still weak
1. Train acoustic model longer.
2. Lower batch size if Kaggle memory is tight.
3. Increase `acoustic_epochs`.
4. Keep `teacher_forcing_ratio` high in early training.
5. Train the vocoder longer than the acoustic model.
6. Do not judge quality too early.

### Practical expectations
- The **acoustic model** may start giving visible alignments before audio sounds good.
- The **vocoder** often needs many epochs before audio becomes clearly hearable.
- If alignment is broken, fix the acoustic model first before blaming the vocoder.